# M1｜從商業問題到預測任務（2h）

**情境**：你是 Olist 的資料科學家。客服團隊每天只有人力主動關懷 **5%** 的新訂單。
主管問：「能不能在顧客**剛下單**時，就找出最可能給差評的訂單？」

本單元產出：
1. 分析問題定義書（決策情境、T0、目標、指標、驗證設計）
2. Data Card 重點：粒度、join 風險、哪些欄位在 T0 之後才出現
3. 兩個 baseline 分數（Dummy、raw LGBM）

> 🤖 **AI 協作方式**：Claude Code 使用者可以直接說「用 problem-framing skill 幫我…」或呼叫 subagent `eda-analyst`；
> Codex 使用者說「依照 .agents/skills/problem-framing/SKILL.md 的流程…」。
> **規則**：AI 產生的每個數字，你都要能指出是哪一行程式算出來的。

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from olist_course import config

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

## 1. 讀資料：9 張表、各自的粒度

In [ ]:
from olist_course.data.load import load_tables

T = load_tables()
pd.DataFrame({name: {"rows": len(df), "cols": df.shape[1]} for name, df in T.items()}).T

### 練習 1：粒度與 join 風險
一張表「一列代表什麼」決定了 join 後會不會重複計算。請檢查：
- `reviews`：一筆訂單會有幾筆評價？
- `items`：一筆訂單會有幾列明細？
- `geolocation`：一個郵遞區號前綴有幾列？

In [ ]:
# TODO：用 groupby(...).size() 算出三張表每個鍵的列數分布
grain = ...
grain

✍️ **你的答案：**



## 2. 預測時點 T0：下單那一秒，你知道什麼？

這是整門課最重要的一個決定。我們選 **T0 = `order_purchase_timestamp`**。

### 練習 2：把所有日期欄位分成「T0 已知 / T0 之後才知道」，並和 `config.POST_T0_COLUMNS` 對照

In [ ]:
# TODO：列出所有 datetime 欄位，標示是否在 config.POST_T0_COLUMNS 中

✍️ **你的答案：**



## 3. 建立訂單層級 base table 與目標變數

In [ ]:
from olist_course.data.build_base import build_base_table
from olist_course.split import assign_split

base = build_base_table(T)
split = assign_split(base[config.T0_COL])
print(f"rows={len(base):,}  low_review_rate={base[config.TARGET].mean():.3f}")
base.groupby(split)[config.TARGET].agg(["size", "mean"])

### 練習 3：為什麼主指標不是 Accuracy？
算出「全部猜不會差評」的 Accuracy，並說明它對客服的決策有什麼用。

In [ ]:
# TODO：算出永遠預測 0 的 Accuracy

✍️ **你的答案：**



## 4. Baseline：先知道「什麼都不做」有多少分

所有實驗都透過 `run_experiment()` 執行：它固定使用 3 個月份 fold 的時間序列 CV，並自動寫入 `experiments/log.jsonl`。

In [ ]:
from olist_course.experiment import compare, run_experiment

r_dummy = run_experiment("raw", model="dummy", note="M1 baseline: prior")
r_raw = run_experiment("raw", model="lgbm", note="M1 baseline: raw LGBM")
r_dummy.folds[["valid_month", "base_rate", "pr_auc"]]

In [ ]:
from dataclasses import replace

compare([replace(r_dummy, name="dummy"), r_raw], baseline="dummy")[
    ["model", "n_features", "pr_auc_mean", "pr_auc_std", "p_at_k_mean", "lift_at_k_mean", "folds_improved"]
]

### 練習 4：寫出你的分析問題定義書（用 problem-framing skill 的模板）

✍️ **你的答案：**

